# Generate the held-out DPO candidate bank

Generate candidates from the split-trained Qwen2.5-0.5B SFT model on only the 226 questions reserved for DPO. The SFT model never trained on these question IDs. The default uses 10 samples per question at temperature 0.7 and the same strict-extractive prompt used for SFT.

The detected completed SFT run used LoRA rank 32 and alpha 32. Its selected `adapter_best_evidence_mrr` checkpoint is used below.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'cse_dpo/generate_candidate_bank.py').is_file():
            return candidate
    raise FileNotFoundError('Could not find the project root')

PROJECT_ROOT = find_project_root(Path.cwd())
SPLIT_DIR = PROJECT_ROOT / 'data/BioASQ_factoid_sft_prepared/gold_supported_question_split_sft80_dpo20_seed3407'
DPO_INPUT = SPLIT_DIR / 'dpo_questions_for_candidate_generation.json'
SFT_IDS_PATH = SPLIT_DIR / 'sft_question_ids.json'
DPO_IDS_PATH = SPLIT_DIR / 'dpo_question_ids.json'
MODEL_ROOT = PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_sft80_dpo20_seed3407_lora_r32_alpha32_dropout_005_strict_extractive'
MODEL_REF = MODEL_ROOT / 'adapter_best_evidence_mrr'
BEST_SUMMARY = MODEL_ROOT / 'evidence_mrr_selection/best_summary.json'

SAMPLES_PER_QUESTION = 10
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 3407
MAX_SEQ_LENGTH = 9000
MAX_NEW_TOKENS = 64
BATCH_SIZE = 1
SKIP_IF_COMPLETE = True
CUSTOM_OUTPUT_ROOT = None  # Example: PROJECT_ROOT / 'Artifacts/cse_dpo/candidate_banks/my_new_bank'
TEMPERATURE_TAG = str(TEMPERATURE).replace('.', '')
DEFAULT_OUTPUT_ROOT = PROJECT_ROOT / f'Artifacts/cse_dpo/candidate_banks/qwen25_05b_sft80_dpo20_r32_step175_dpo226_s{SAMPLES_PER_QUESTION}_t{TEMPERATURE_TAG}_seed{SEED}'
OUTPUT_ROOT = Path(CUSTOM_OUTPUT_ROOT) if CUSTOM_OUTPUT_ROOT else DEFAULT_OUTPUT_ROOT
if not OUTPUT_ROOT.is_absolute():
    OUTPUT_ROOT = PROJECT_ROOT / OUTPUT_ROOT

MODEL_SLUG = 'adapter-best-evidence-mrr'
BANK_DIR = OUTPUT_ROOT / MODEL_SLUG
BANK_PATH = BANK_DIR / 'candidate_bank.jsonl'
MANIFEST_PATH = BANK_DIR / 'manifest.json'


In [ ]:
required = [
    DPO_INPUT, SFT_IDS_PATH, DPO_IDS_PATH,
    MODEL_REF / 'adapter_config.json', BEST_SUMMARY,
    PROJECT_ROOT / 'prompts/factoid_single_answer_aligned.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required files: {missing}'

dpo_rows = json.loads(DPO_INPUT.read_text(encoding='utf-8'))
sft_ids = set(json.loads(SFT_IDS_PATH.read_text(encoding='utf-8'))['question_ids'])
dpo_ids = json.loads(DPO_IDS_PATH.read_text(encoding='utf-8'))['question_ids']
adapter_config = json.loads((MODEL_REF / 'adapter_config.json').read_text(encoding='utf-8'))
best_summary = json.loads(BEST_SUMMARY.read_text(encoding='utf-8'))

assert len(dpo_rows) == len(dpo_ids) == 226
assert [row['id'] for row in dpo_rows] == dpo_ids
assert not (sft_ids & set(dpo_ids))
assert adapter_config['r'] == 32 and adapter_config['lora_alpha'] == 32, adapter_config
assert best_summary['step'] == 175, best_summary

print(json.dumps({
    'model': str(MODEL_REF),
    'selected_step': best_summary['step'],
    'selected_dev_mrr': best_summary['metric_value'],
    'lora_r': adapter_config['r'],
    'lora_alpha': adapter_config['lora_alpha'],
    'dpo_questions': len(dpo_rows),
    'samples_per_question': SAMPLES_PER_QUESTION,
    'expected_candidates': len(dpo_rows) * SAMPLES_PER_QUESTION,
    'output': str(BANK_PATH),
}, indent=2))

## Generate

This generates 2,260 candidate records by default. Set `SAMPLES_PER_QUESTION=20` before running if you need exact sample-count comparability with the older banks.

In [ ]:
def bank_is_complete() -> bool:
    if not BANK_PATH.exists() or not MANIFEST_PATH.exists():
        return False
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    row_count = sum(1 for line in BANK_PATH.open(encoding='utf-8') if line.strip())
    expected = len(dpo_rows) * SAMPLES_PER_QUESTION
    generation = manifest.get('generation', {})
    return (
        manifest.get('dataset', {}).get('question_count') == len(dpo_rows)
        and manifest.get('dataset', {}).get('eval_input') == [str(DPO_INPUT.resolve())]
        and manifest.get('model', {}).get('ref') == str(MODEL_REF)
        and manifest.get('prompt', {}).get('prompt_id') == 'factoid-single-answer-extractive-v1'
        and generation.get('samples_per_question_total') == SAMPLES_PER_QUESTION
        and generation.get('temperature') == TEMPERATURE
        and generation.get('top_p') == TOP_P
        and generation.get('seed') == SEED
        and manifest.get('candidate_count') == expected
        and row_count == expected
    )

command = [
    sys.executable, '-m', 'cse_dpo.generate_candidate_bank',
    '--eval-input', str(DPO_INPUT),
    '--model-ref', str(MODEL_REF),
    '--output-dir', str(OUTPUT_ROOT),
    '--dataset-name', 'bioasq-factoid-train-dpo-holdout',
    '--question-types', 'factoid',
    '--prompt-file', 'prompts/factoid_single_answer_aligned.json',
    '--prompt', 'factoid-single-answer-extractive-v1',
    '--chat-template', 'qwen-2.5',
    '--prompt-format', 'chat',
    '--samples-per-question-total', str(SAMPLES_PER_QUESTION),
    '--max-resources', '0',
    '--max-resource-chars', '0',
    '--resource-selection', 'first',
    '--resource-granularity', 'document',
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--temperature', str(TEMPERATURE),
    '--top-p', str(TOP_P),
    '--batch-size', str(BATCH_SIZE),
    '--seed', str(SEED),
    '--factoid-parser-mode', 'agnostic',
    '--local-files-only',
]

print(' '.join(command))
if SKIP_IF_COMPLETE and bank_is_complete():
    print('Candidate bank is already complete; skipping generation.')
else:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    assert bank_is_complete(), 'Generation finished but the bank failed completeness validation.'
print('Candidate bank:', BANK_PATH)

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
print(json.dumps({
    'candidate_bank': str(BANK_PATH),
    'question_count': manifest['dataset']['question_count'],
    'candidate_count': manifest['candidate_count'],
    'samples_per_question': manifest['generation']['samples_per_question_total'],
    'generator': manifest['model'],
}, indent=2))